In [1]:
import pandas as pd

df = pd.read_csv('maindataset.csv', on_bad_lines='skip')

print(df.columns.tolist())
print(df.head(3))

FileNotFoundError: [Errno 2] No such file or directory: 'maindataset.csv'

In [ ]:
import pandas as pd

df = pd.read_csv('maindataset.csv', on_bad_lines='skip')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

df_1yr = df.loc['2017-01-01':'2017-12-31']

df_sample = df_1yr.resample('1h').mean()

df_sample.to_csv('solar_1yr_raw_sample.csv')

print("Raw sample created. Shape:", df_sample.shape)

In [ ]:
import io
import numpy as np
import pandas as pd
try:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
except ImportError:
    df = pd.read_csv("sampledata.csv")

print("Raw shape:", df.shape)
report = {}
df.columns = df.columns.str.strip().str.lower()

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
report["bad_timestamps_dropped"] = int(df["timestamp"].isna().sum())
df = df.dropna(subset=["timestamp"])

report["duplicate_rows_dropped"] = int(df.duplicated().sum())
df = df.drop_duplicates()

report["duplicate_timestamps_dropped"] = int(df["timestamp"].duplicated().sum())
df = df.drop_duplicates(subset="timestamp", keep="first")

df = df.sort_values("timestamp").set_index("timestamp")

df = df.asfreq("h")

empty_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns=empty_cols)
report["empty_columns_dropped"] = empty_cols

def to_nan(mask, label):
    report[label] = int(mask.sum())
    return mask
if "wind_direction" in df:
    df.loc[to_nan(~df["wind_direction"].between(0, 360) & df["wind_direction"].notna(),
                  "wind_direction_out_of_range"), "wind_direction"] = np.nan
if "weather_relative_humidity" in df:
    h = df["weather_relative_humidity"]
    report["humidity_clipped_to_100"] = int((h > 100).sum())
    df["weather_relative_humidity"] = h.clip(0, 100)
if "performance_ratio" in df:
    pr = df["performance_ratio"]
    df.loc[to_nan(~pr.between(0, 100) & pr.notna(), "performance_ratio_out_of_range"),
           "performance_ratio"] = np.nan

for c in ["active_power", "current_phase_average", "weather_daily_rainfall",
          "global_horizontal_radiation", "diffuse_horizontal_radiation",
          "radiation_global_tilted", "radiation_diffuse_tilted"]:
    if c in df:
        report[f"{c}_negatives_set_to_0"] = int((df[c] < 0).sum())
        df[c] = df[c].clip(lower=0)

if "active_energy_delivered_received" in df:
    e = df["active_energy_delivered_received"]
    running_max = e.cummax()
    bad = e < running_max
    report["energy_meter_decreases"] = int(bad.sum())
    df.loc[bad, "active_energy_delivered_received"] = np.nan

for c in ["weather_temperature_celsius", "weather_daily_rainfall"]:
    if c in df:
        q1, q3 = df[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
            mask = (df[c] < lo) | (df[c] > hi)
            report[f"{c}_outliers_flagged"] = int(mask.sum())

            if c == "weather_temperature_celsius":
                df.loc[mask, c] = np.nan
MAX_GAP = 6
before = df.isna().sum().sum()

cont_cols = [c for c in df.columns if c != "wind_direction"]
df[cont_cols] = df[cont_cols].interpolate(method="time", limit=MAX_GAP,
                                          limit_area="inside")
if "wind_direction" in df:
    rad = np.deg2rad(df["wind_direction"])
    s = pd.Series(np.sin(rad), index=df.index).interpolate(method="time", limit=MAX_GAP, limit_area="inside")
    c_ = pd.Series(np.cos(rad), index=df.index).interpolate(method="time", limit=MAX_GAP, limit_area="inside")
    df["wind_direction"] = (np.rad2deg(np.arctan2(s, c_)) % 360).where(s.notna())

report["values_filled_by_interpolation"] = int(before - df.isna().sum().sum())

df["date"] = df.index.date
df["hour"] = df.index.hour
df["month"] = df.index.month

print("\nCleaned shape:", df.shape)
print("\n--- Cleaning report ---")
for k, v in report.items():
    print(f"{k}: {v}")
print("\nRemaining missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])


df.reset_index().to_csv("sampledata_cleaned.csv", index=False)
try:
    files.download("sampledata_cleaned.csv")
except NameError:
    pass
df.head()

In [ ]:
import io
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

try:
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
except ImportError:
    df = pd.read_csv("sampledata_cleaned.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").set_index("timestamp")


df = df.dropna(subset=["active_power"])


candidate_features = [
    "global_horizontal_radiation",
    "diffuse_horizontal_radiation",
    "radiation_global_tilted",
    "radiation_diffuse_tilted",
    "weather_temperature_celsius",
    "weather_relative_humidity",
    "wind_direction",
    "weather_daily_rainfall",
    "hour",
    "month"
]


features = [col for col in candidate_features if col in df.columns]
print("Features used for training:", features)

X = df[features]
y = df["active_power"]

split_idx = int(len(df) * (10 / 12))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")


quantiles = {0.10: "p10", 0.50: "p50", 0.90: "p90"}
preds = {}

for alpha, label in quantiles.items():
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    preds[label] = model.predict(X_test)


results = pd.DataFrame({
    "timestamp": X_test.index,
    "y_true": y_test.values,
    "p10": preds["p10"],
    "p50": preds["p50"],
    "p90": preds["p90"]
}).set_index("timestamp")


results[["p10", "p50", "p90"]] = results[["p10", "p50", "p90"]].clip(lower=0)
results[["p10", "p50", "p90"]] = np.sort(results[["p10", "p50", "p90"]].values, axis=1)


mae = mean_absolute_error(results["y_true"], results["p50"])
rmse = root_mean_squared_error(results["y_true"], results["p50"])


in_interval = (results["y_true"] >= results["p10"]) & (results["y_true"] <= results["p90"])
coverage = in_interval.mean() * 100

print("\n--- Model Evaluation (Test Set) ---")
print(f"MAE  (P50): {mae:.4f} kW")
print(f"RMSE (P50): {rmse:.4f} kW")
print(f"P10-P90 Coverage (PICP): {coverage:.2f}% (Ideal: ~80%)")


sample_plot = results.iloc[:120]
plt.figure(figsize=(12, 5))
plt.plot(sample_plot.index, sample_plot["y_true"], label="Actual Power", color="black", linewidth=1.5)
plt.plot(sample_plot.index, sample_plot["p50"], label="Forecast (P50)", color="blue", linestyle="--")
plt.fill_between(sample_plot.index, sample_plot["p10"], sample_plot["p90"],
                 color="skyblue", alpha=0.4, label="Uncertainty Band (P10-P90)")
plt.title("Probabilistic Solar Power Forecast (5-Day Test Sample)")
plt.ylabel("Active Power (kW)")
plt.xlabel("Timestamp")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


results.reset_index().to_csv("predictions.csv", index=False)
try:
    files.download("predictions.csv")
except NameError:
    pass

In [ ]:
!pip install -q shap

import shap
import lightgbm as lgb
import matplotlib.pyplot as plt


model_p50 = lgb.LGBMRegressor(
    objective="quantile",
    alpha=0.50,
    n_estimators=200,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)
model_p50.fit(X_train, y_train)


explainer = shap.TreeExplainer(model_p50)
shap_values = explainer(X_test)


plt.figure(figsize=(8, 5))
shap.plots.bar(shap_values, show=False)
plt.title("SHAP Global Feature Importance (P50 Model)")
plt.tight_layout()
plt.savefig("shap_bar.png", dpi=150)
plt.show()


plt.figure(figsize=(8, 5))
shap.plots.beeswarm(shap_values, show=False)
plt.title("SHAP Beeswarm Plot (Feature Impact on Solar Power)")
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np

# Bin edges come from the TRAINING data only (avoids leakage)
daytime = y_train[y_train > 0]
q1, q2 = daytime.quantile([0.33, 0.66])
bins = [-np.inf, 0.0, q1, q2, np.inf]
labels = ["Zero (night)", "Low", "Medium", "High"]

def to_class(s):
    return pd.cut(s, bins=bins, labels=labels).astype(str)

y_true_cls = to_class(results["y_true"])
y_pred_cls = to_class(results["p50"])

cm = confusion_matrix(y_true_cls, y_pred_cls, labels=labels)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion Matrix (counts)")

cm_norm = confusion_matrix(y_true_cls, y_pred_cls, labels=labels, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=labels).plot(ax=ax[1], cmap="Blues",
                                                           colorbar=False, values_format=".2f")
ax[1].set_title("Confusion Matrix (row-normalized)")

for a in ax:
    a.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

print(classification_report(y_true_cls, y_pred_cls, labels=labels, zero_division=0))